# MuseTalk 1.5 GPU validation worker

T4 validation worker: approved traditional-clothing singer image/video + successful ACE-Step bhajan audio -> MuseTalk 1.5 lip-synced MP4. Do not publish until visual QA passes.

In [ ]:
# Runtime/repository setup
!nvidia-smi
!pip -q install uv
import subprocess
from pathlib import Path
MT=Path('/content/MuseTalk')
VENV=Path('/content/musetalk310')
if not MT.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk.git',str(MT)],check=True)
if not (VENV/'bin/python').exists(): subprocess.run(['uv','venv','--python','3.10',str(VENV)],check=True)
PY=str(VENV/'bin/python')
subprocess.run([PY,'-V'],check=True)


In [ ]:
# Python 3.10 dependencies
import subprocess
PY='/content/musetalk310/bin/python'
MT='/content/MuseTalk'
subprocess.run(['uv','pip','install','--python',PY,'pip','setuptools','wheel'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'torch==2.0.1','torchvision==0.15.2','torchaudio==2.0.2','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'-r',MT+'/requirements.txt'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'openmim','gdown'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'--no-build-isolation','chumpy==0.70'],check=True)
MIM='/content/musetalk310/bin/mim'
for pkg in ['mmengine','mmcv==2.0.1','mmdet==3.1.0','mmpose==1.1.0']: subprocess.run([MIM,'install',pkg],check=True)
print('MuseTalk dependencies installed.')


In [ ]:
# Resumable component downloads. Never rerun the all-in-one download_weights.sh.
import subprocess
from pathlib import Path
MODELS=Path('/content/MuseTalk/models'); MODELS.mkdir(parents=True,exist_ok=True)
def hf(repo,name,target,min_bytes):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>=min_bytes: print('✓',target,'already ready'); return
    url=f'https://huggingface.co/{repo}/resolve/main/{name}?download=true'
    print('Downloading',url)
    subprocess.run(['curl','-L','--fail','--retry','8','--retry-delay','5','-C','-','-o',str(target),url],check=True)
    if target.stat().st_size<min_bytes: raise RuntimeError(f'Incomplete download: {target}')
hf('TMElyralab/MuseTalk','musetalkV15/unet.pth',MODELS/'musetalkV15/unet.pth',3_000_000_000)
hf('TMElyralab/MuseTalk','musetalkV15/musetalk.json',MODELS/'musetalkV15/musetalk.json',500)
hf('stabilityai/sd-vae-ft-mse','config.json',MODELS/'sd-vae/config.json',500)
hf('stabilityai/sd-vae-ft-mse','diffusion_pytorch_model.bin',MODELS/'sd-vae/diffusion_pytorch_model.bin',300_000_000)
hf('openai/whisper-tiny','config.json',MODELS/'whisper/config.json',500)
hf('openai/whisper-tiny','preprocessor_config.json',MODELS/'whisper/preprocessor_config.json',1000)
hf('openai/whisper-tiny','pytorch_model.bin',MODELS/'whisper/pytorch_model.bin',100_000_000)
print('Core weights ready.')


In [ ]:
# Face parsing weights
import subprocess
from pathlib import Path
FP=Path('/content/MuseTalk/models/face-parse-bisent'); FP.mkdir(parents=True,exist_ok=True)
g=FP/'79999_iter.pth'
if not g.exists() or g.stat().st_size<100_000_000: subprocess.run(['/content/musetalk310/bin/gdown','--id','154JgKpzCPW82qINcVieuPH3fZ2e0P812','-O',str(g)],check=True)
r=FP/'resnet18-5c106cde.pth'
if not r.exists() or r.stat().st_size<1_000_000: subprocess.run(['curl','-L','--fail','--retry','8','-C','-','-o',str(r),'https://download.pytorch.org/models/resnet18-5c106cde.pth'],check=True)
print('Face parsing weights ready.')


In [ ]:
# Upload approved singer image/video + successful ACE-Step audio
from google.colab import files
from pathlib import Path
print('Upload the APPROVED traditional-clothing singer image/video:')
avatar=next(iter(files.upload()))
print('Upload the successful ACE-Step bhajan MP3/WAV:')
audio=next(iter(files.upload()))
assert Path(avatar).suffix.lower() in {'.png','.jpg','.jpeg','.webp','.mp4','.mov','.webm'}, f'Unsupported avatar: {avatar}'
assert Path(audio).suffix.lower() in {'.mp3','.wav','.m4a','.flac','.aac','.ogg'}, f'Please upload the actual audio file, not a ZIP: {audio}'
print('Avatar:',avatar); print('Audio:',audio)


In [ ]:
# Normalize inputs
import subprocess
from pathlib import Path
OUT=Path('/content/musetalk_output'); OUT.mkdir(exist_ok=True)
avatar_src=OUT/'avatar_source.png'; audio_wav=OUT/'audio.wav'
subprocess.run(['ffmpeg','-y','-i',avatar,'-frames:v','1','-vf','scale=512:-2','-pix_fmt','rgb24',str(avatar_src)],check=True)
subprocess.run(['ffmpeg','-y','-i',audio,'-ar','16000','-ac','1',str(audio_wav)],check=True)
assert avatar_src.stat().st_size>10000 and audio_wav.stat().st_size>10000
print('Avatar:',avatar_src); print('Audio:',audio_wav)


In [ ]:
# Create MuseTalk task config
from pathlib import Path
MT=Path('/content/MuseTalk'); OUT=Path('/content/musetalk_output')
cfg=MT/'configs/inference/test.yaml'
cfg.write_text(f'''bhajan_test:
  video_path: "{OUT/'avatar_source.png'}"
  audio_path: "{OUT/'audio.wav'}"
  result_name: "bhajan_lipsync.mp4"
''')
print(cfg.read_text())


In [ ]:
# MuseTalk 1.5 inference
# Force headless matplotlib: Colab's matplotlib_inline backend breaks MMPose imports.
import os,subprocess
os.chdir('/content/MuseTalk')
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['PYTHONPATH']='/content/MuseTalk:'+env.get('PYTHONPATH','')
cmd=['/content/musetalk310/bin/python','-m','scripts.inference','--inference_config','configs/inference/test.yaml','--result_dir','/content/musetalk_output/result','--unet_model_path','models/musetalkV15/unet.pth','--unet_config','models/musetalkV15/musetalk.json','--whisper_dir','models/whisper','--version','v15','--fps','25','--batch_size','4','--use_float16','--parsing_mode','jaw']
print('Starting MuseTalk 1.5 on T4...')
result=subprocess.run(cmd,env=env,text=True,capture_output=True)
print(result.stdout)
if result.returncode!=0: print(result.stderr); raise RuntimeError(f'MuseTalk failed with exit code {result.returncode}')
print('MUSE TALK FINISHED')


In [ ]:
# Download final MP4
from pathlib import Path
from google.colab import files
result_dir=Path('/content/musetalk_output/result'); candidates=list(result_dir.rglob('*.mp4'))
assert candidates,'MuseTalk produced no MP4'
candidate=max(candidates,key=lambda p:p.stat().st_size)
assert candidate.stat().st_size>100000
print('SUCCESS:',candidate); print('Size:',round(candidate.stat().st_size/1024/1024,2),'MB')
files.download(str(candidate))
